In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_chroma.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.retrievers.bm25 import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
import numpy as np
import pandas as pd
import json
import re

In [2]:
document_path = r"D:\computer_science_research_papers\Artificial_Intelligence\02_General Quantification of Covariate and Concept Shifts.pdf"
raw_docs = PyMuPDFLoader(document_path).load()

In [3]:
for doc in raw_docs[:3]:
    print(doc.metadata['page'])
    print(json.dumps(doc.page_content))

0
"General Quantification of Covariate and Concept Shifts\nHongbo Chen 1 Li Charlie Xia 1\nAbstract\nGeneralization under distribution shift remains a\ncore challenge in modern machine learning, yet\nexisting learning bound theory is limited to nar-\nrow, idealized settings and is non-estimable from\nsamples. In this paper, we bridge the gap between\ntheory and practical applications. We first show\nthat existing definition of concept shift breaks\nwhen the source and target supports mismatch.\nLeveraging entropic optimal transport, we pro-\npose a key notion: \u03b3\u2217-concept shifts, and derive\na general error bound unifying covariate and \u03b3\u2217-\nconcept shifts, which applies to broad loss func-\ntions, label spaces, and stochastic labeling. We\nfurther develop estimators for these shifts with\nconcentration guarantees, and the DataShifts algo-\nrithm, which can quantify distribution shifts and\nestimate the error bound in most applications - a\nrigorous and general tool f

In [4]:
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    return text


In [5]:
docs = [clean_text(doc.page_content) for doc in raw_docs]

In [6]:
docs[23]

'general quantification of covariate and concept shifts\nwhere\nv a \n\na2  1 a\n\na2  2\na  1\na2  2a 1 a\n\na2  4a 2\na 1\ndefine vε  v\ncε\n\n v\nw1µ νε\n\n as shown in the range of v a derivation vε 2 \n\n3 2 and depends only\non cε\nwith this choice t  tc ε  t  4cε0  8ε2\n0 hence\np\nw deb\n1\nc  ε\n\np\nt  4cε0  8ε2\n0\n\n2 exp\n\nλsnsε2\n0\n4\n\n 2 exp\n\nλt nt ε2\n0\n4\n\n 2 exp\n\nλsnsvεε2\n32\n\n 2 exp\n\nλt nt vεε2\n32\n\n\nfinally note c  w1µ ν  wβds\nx dt\nx when β  0 and w deb\n1\n w deb\nβ\nwhen β  0 so the above inequality is\nexactly 10 this completes the proof\nrange of v a\nrecall\nv a \n\nv1a  a2  1 a\n\na2  2\n0 a  1\nv2a  a2  2a 1 a\n\na2  4a 2\na 1\nfirst the two branches agree at a  1\nv11  2 \n\n3\nv21  2 \n\n3\ni monotonicity on 0 1 differentiate v1 on 0 1\nv \n1a  2a \np\na2  2 \na2\n\na2  2\n 2a 2a2  1\n\na2  2\n\nfor a 0 we have a\n\na2  2 a2  1 since\na2  12 a2a2  2  1  0\nthus\na21\n\na22 a which implies v \n1a 0 on 0 1 hence v1 is decreasing on 0 1 so\nv

In [7]:
tf_idf_vector = TfidfVectorizer()

In [8]:
tf_idf_vector_docs = tf_idf_vector.fit_transform(docs).toarray()

In [9]:
tf_idf_vector_docs[0]

array([0., 0., 0., ..., 0., 0., 0.], shape=(2565,))

In [10]:
query = "Entropic Optimal Transport"

In [11]:
tf_idf_vector_query = tf_idf_vector.transform([query]).toarray()

In [12]:
simillarity = cosine_similarity(tf_idf_vector_docs, tf_idf_vector_query)

In [13]:
index_mapping = dict(enumerate(np.argsort(simillarity, axis=0).flatten()))
index_mapping = dict(sorted(index_mapping.items(), key=lambda x: x[1]))

print(index_mapping)

{23: np.int64(0), 31: np.int64(1), 37: np.int64(2), 34: np.int64(3), 33: np.int64(4), 36: np.int64(5), 27: np.int64(6), 0: np.int64(7), 21: np.int64(8), 35: np.int64(9), 25: np.int64(10), 1: np.int64(11), 28: np.int64(12), 17: np.int64(13), 2: np.int64(14), 30: np.int64(15), 32: np.int64(16), 14: np.int64(17), 7: np.int64(18), 26: np.int64(19), 4: np.int64(20), 5: np.int64(21), 6: np.int64(22), 3: np.int64(23), 15: np.int64(24), 29: np.int64(25), 24: np.int64(26), 16: np.int64(27), 18: np.int64(28), 10: np.int64(29), 8: np.int64(30), 9: np.int64(31), 11: np.int64(32), 19: np.int64(33), 22: np.int64(34), 12: np.int64(35), 13: np.int64(36), 20: np.int64(37)}


In [14]:
for doc_index, value in list(index_mapping.items())[:3]:
    print("rank:", value)
    print(docs[doc_index][:100])


rank: 0
general quantification of covariate and concept shifts
where
v a 

a2  1 a

a2  2
a  1
a2  2a 1 a

a
rank: 1
general quantification of covariate and concept shifts
an upper bound for each pair xs xt 
fix xs xt
rank: 2
general quantification of covariate and concept shifts
let hbe the optimal value and define lsdp 

h


In [15]:
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
    cache_folder='.models',
    model_kwargs={"device": "cpu", "local_files_only": False},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 428.75it/s]


In [16]:
chroma_db = Chroma(
    collection_name='hybride_search',
    embedding_function=embedding_model, 
    persist_directory='.chroma/hybride_search'
)

In [23]:
docs = [Document(page_content=content) for content in docs]

In [25]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
)
chunks = splitter.split_documents(docs)

In [ ]:
chroma_db.add_documents(chunks)

In [28]:
vector_store_results = chroma_db.similarity_search_with_relevance_scores(query = query, k=5)

In [51]:
chrom_db_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 5}
)

In [52]:
chrom_db_retriever.invoke(input=query)

[Document(id='b21ac935-2405-473d-884c-ecc01f8ebf91', metadata={}, page_content='23 entropic optimal transport\nwe introduce entropic optimal transport which we will em\nploy to give the general definitions of x and yx shifts'),
 Document(id='2bdad3ba-0752-4a41-8691-eb9b4f979be4', metadata={}, page_content='cambridge university press 2004\ncarlier g duval v peyre g and schmitzer b con\nvergence of entropic schemes for optimal transport and\ngradient flows siam journal on mathematical analysis'),
 Document(id='8b35824b-210d-425e-b38b-ddf22be173fa', metadata={}, page_content='suppγ suppds\nx  suppdt\nx\nc4 proof of lemma 35\nproof let ν  ds\nx dt\nx and consider the entropic optimal transport objective\njγ \nz\nρx xs xt  dγxs xt   β hγ  ν\nγ γds\nx dt\nx'),
 Document(id='5bcc188e-bf70-4717-809d-b5ad3d9eb2b5', metadata={}, page_content='distributions p and q on the metric space ω ρ and a pa\nrameter β 0 the order1 entropic optimal transport is\nwβp q \ninf\nγγpq\nz\nρ dγ  β h\nγ\n\x0c\x0c 

In [55]:
bm25 = BM25Retriever.from_documents(chunks, k=5)

In [56]:
bm25.invoke(input = query)

[Document(metadata={}, page_content='increases and overestimation appears but it stays below the bound set by irreducible error as proved in proposition 47\n38'),
 Document(metadata={}, page_content='and the ˆscpt still does not notably overestimate as σ grows far beyond 10 yx shift becomes less identifiable the bias'),
 Document(metadata={}, page_content='sγ\ncpt  10 even when the noise in both domains reaches half of the true yx shift σ  05 the bias ˆremains small'),
 Document(metadata={}, page_content='20\n40\n100\n200\nbias  ˆ\n00036\n00389\n00702\n01440\n04865\n14524\n47314\n103467\nwhen the noise σ is small the bias ˆis also very small which means our estimator ˆscpt stays close to the true yx shift\nsγ'),
 Document(metadata={}, page_content='report the average ˆ the results are shown below\ntable 3 bias of ˆscpt under different noise levels σ\nσ\n001\n01\n03\n05\n10\n20\n50\n100\nirreducible error bound 2σ\n002\n02\n06\n10\n20\n40\n100\n200\nbias  ˆ\n00036')]

In [36]:
query

'Entropic Optimal Transport'

In [57]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[chrom_db_retriever, bm25],
    weights=[0.7, 0.3], 
)

In [58]:
ensemble_retriever.invoke(input=query, k=4)

[Document(id='b21ac935-2405-473d-884c-ecc01f8ebf91', metadata={}, page_content='23 entropic optimal transport\nwe introduce entropic optimal transport which we will em\nploy to give the general definitions of x and yx shifts'),
 Document(id='2bdad3ba-0752-4a41-8691-eb9b4f979be4', metadata={}, page_content='cambridge university press 2004\ncarlier g duval v peyre g and schmitzer b con\nvergence of entropic schemes for optimal transport and\ngradient flows siam journal on mathematical analysis'),
 Document(id='8b35824b-210d-425e-b38b-ddf22be173fa', metadata={}, page_content='suppγ suppds\nx  suppdt\nx\nc4 proof of lemma 35\nproof let ν  ds\nx dt\nx and consider the entropic optimal transport objective\njγ \nz\nρx xs xt  dγxs xt   β hγ  ν\nγ γds\nx dt\nx'),
 Document(id='5bcc188e-bf70-4717-809d-b5ad3d9eb2b5', metadata={}, page_content='distributions p and q on the metric space ω ρ and a pa\nrameter β 0 the order1 entropic optimal transport is\nwβp q \ninf\nγγpq\nz\nρ dγ  β h\nγ\n\x0c\x0c 